In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [ ]:
# Define paths
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
networks_path = base_path / "Processed_data/networks"
damages_path = base_path / "Processed_data/direct_damages_summary_uids"
hydrobasins_path = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"

# Define subfolders for networks
subfolders = ["energy", "transport", "buildings", "water"]

# Define Jamaica CRS
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
# Load HydroBASINS data
hydrobasins = gpd.read_file(hydrobasins_path)

# Ensure it uses Jamaica's CRS
hydrobasins = hydrobasins.to_crs(jamaica_metric_grid_crs)
print(f"Loaded Hydrobasins with {len(hydrobasins)} polygons.")

In [ ]:
# Load network data
network_data = {}

for subfolder in subfolders:
    folder_path = networks_path / subfolder
    gpkg_files = folder_path.glob("*.gpkg")
    
    for gpkg_file in gpkg_files:
        try:
            data = gpd.read_file(gpkg_file)
            # Reproject if necessary
            if data.crs != jamaica_metric_grid_crs:
                data = data.to_crs(jamaica_metric_grid_crs)
            # Create a unique key using subfolder and file stem
            key = f"{subfolder}_{gpkg_file.stem}"
            network_data[key] = data
            print(f"Loaded and reprojected: {gpkg_file.name}")
        except Exception as e:
            print(f"Failed to load {gpkg_file.name}: {e}")

# Confirmation
print(f"Successfully loaded {len(network_data)} network layers.")

# # Function to standardize keys for matching
# def standardize_key(name):
#     """Standardize keys by taking the first part before an underscore."""
#     return name.split("_")[0]



In [ ]:
# # Load damages data
# damages_data = {}

# # Filter for fluvial hazards
# parquet_files = damages_path.glob("*.parquet")

# for file in parquet_files:
#     data = pd.read_parquet(file)
#     if 'hazard' in data.columns:
#         fluvial_data = data[data['hazard'] == 'fluvial']
#         damages_data[file.stem] = fluvial_data
#         print(f"Loaded and filtered: {file.name} with {len(fluvial_data)} fluvial rows.")
#     else:
#         print(f"No 'hazard' column found in: {file.name}")

In [ ]:
# Load damages data (EAEL only)
damages_data = {}

# Filter for fluvial hazards in EAEL files
parquet_files = damages_path.glob("*.parquet")

for file in parquet_files:
    if "EAEL" in file.stem:  # Only process files with "EAEL" in the name
        data = pd.read_parquet(file)
        if 'hazard' in data.columns:
            fluvial_data = data[data['hazard'] == 'fluvial']
        # Skip files with no fluvial data
            if len(fluvial_data) == 0:
                print(f"File {file.name} has no fluvial rows.")
                continue
                
            # In EAEL loading
        
            # Standardize key by removing suffixes
            # Standardize key by removing suffixes
            standardized_key = standardize_key(file.stem)  # e.g., 'rail'
            damages_data[standardized_key] = fluvial_data
            print(f"Loaded and filtered (EAEL): {file.name} with {len(fluvial_data)} fluvial rows.")
        else:
            print(f"No 'hazard' column found in (EAEL): {file.name}")
            # In EAEL loading


# Standardize network keys for matching
network_data = {standardize_key(key): value for key, value in network_data.items()}

# Check keys for damages_data and network_data
print("\nKeys in damages_data:")
print(damages_data.keys())

print("\nKeys in network_data:")
print(network_data.keys())


# Summary of loaded data
print("\nSummary:")
print(f"Total Hydrobasins loaded: {len(hydrobasins)} polygons")
print(f"Total network layers loaded: {len(network_data)}")
print(f"Total EAEL damage datasets loaded: {len(damages_data)}")

In [ ]:
# Print columns for a sample network GeoDataFrame
for network_name, network_gdf in network_data.items():
    print(f"Columns in network_data[{network_name}]: {network_gdf.columns}")

# Print columns for a sample damages DataFrame
for damage_name, damages_df in damages_data.items():
    print(f"Columns in damages_data[{damage_name}]: {damages_df.columns}")

In [ ]:
# Match damages to network layers
joined_networks = {}

for network_name, network_gdf in network_data.items():
    base_network_name = network_name.split("_")[-1]  # Extract file stem
    if base_network_name in damages_data:
        damages_df = damages_data[base_network_name]

        # Identify the correct join key
        join_key = None
        for col in ['edge_id', 'node_id', 'osm_id', 'id', 'uid']:
            if col in network_gdf.columns and col in damages_df.columns:
                join_key = col
                break

        if join_key:
            # Perform the join
            print(f"Joining {network_name} on key '{join_key}'")
            joined = network_gdf.merge(damages_df, on=join_key, how="inner")
            joined_networks[network_name] = joined
            print(f"Joined {network_name} with EAEL damages. Result: {len(joined)} rows.")
        else:
            print(f"No common key found for {network_name}. Skipping...")
    else:
        print(f"No matching EAEL damages found for {network_name}.")

In [ ]:
print("Keys in network_data:")
print(network_data.keys())

In [ ]:
potable_facilities_data = pd.read_parquet(file_path, engine="pyarrow")
print(potable_facilities_data.columns)